# 🚀 Model Deployment + Git/GitHub + CI/CD
---
## Deployment Pipeline:
```
Train Model → Save (pickle) → API (Flask/FastAPI) → Docker → Cloud → Monitor
```

## Git Essentials:
```
git init → git add → git commit → git push → git pull
```

## CI/CD:
```
Push code → Auto-test → Auto-deploy (if tests pass)
```

In [ ]:
# ============================================================================
# 1. SAVE & LOAD MODEL
# ============================================================================
import pickle, joblib
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

print('1️⃣ SAVE & LOAD MODEL')
print('=' * 50)

# Train a quick model
X, y = make_classification(n_samples=500, n_features=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
print(f'Original accuracy: {accuracy_score(y_test, model.predict(X_test)):.4f}')

# Method 1: pickle
with open('model_pickle.pkl', 'wb') as f:
    pickle.dump(model, f)
print('✅ Saved with pickle: model_pickle.pkl')

# Method 2: joblib (better for large numpy arrays)
joblib.dump(model, 'model_joblib.pkl')
print('✅ Saved with joblib: model_joblib.pkl')

# Load and verify
loaded = joblib.load('model_joblib.pkl')
print(f'Loaded accuracy: {accuracy_score(y_test, loaded.predict(X_test)):.4f}')
print('✅ Matches! Model saved correctly.')

## 2️⃣ FLASK API FOR ML MODEL

```python
# app.py — Run with: python app.py
from flask import Flask, request, jsonify
import joblib
import numpy as np

app = Flask(__name__)
model = joblib.load('model_joblib.pkl')

@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.get_json()
        features = np.array(data['features']).reshape(1, -1)
        prediction = int(model.predict(features)[0])
        probability = float(model.predict_proba(features)[0][1])
        return jsonify({
            'prediction': prediction,
            'probability': round(probability, 4),
            'status': 'success'
        })
    except Exception as e:
        return jsonify({'error': str(e), 'status': 'failed'}), 400

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'Model is running!'})

if __name__ == '__main__':
    app.run(debug=True, port=5000)
```

**Test with curl:**
```bash
curl -X POST http://localhost:5000/predict \
  -H "Content-Type: application/json" \
  -d '{"features": [1.2, 3.4, 5.6, 7.8, 9.0, 1.1, 2.2, 3.3, 4.4, 5.5]}'
```

## 3️⃣ FASTAPI (Modern, Faster Alternative)

```python
# main.py — Run with: uvicorn main:app --reload
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np

app = FastAPI(title='ML Prediction API')
model = joblib.load('model_joblib.pkl')

class InputData(BaseModel):
    features: list[float]

@app.post('/predict')
def predict(data: InputData):
    features = np.array(data.features).reshape(1, -1)
    prediction = int(model.predict(features)[0])
    probability = float(model.predict_proba(features)[0][1])
    return {
        'prediction': prediction,
        'probability': round(probability, 4)
    }

@app.get('/')
def root():
    return {'message': 'ML API is running! Visit /docs for UI'}
```

**Why FastAPI over Flask?**
- Auto-generated docs at `/docs`
- Async support (faster)
- Type validation with Pydantic
- Better for production ML APIs

## 4️⃣ DOCKER (Containerize Your Model)

```dockerfile
# Dockerfile
FROM python:3.10-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install -r requirements.txt

COPY . .

EXPOSE 5000

CMD ["python", "app.py"]
```

```txt
# requirements.txt
flask==3.0.0
scikit-learn==1.3.0
joblib==1.3.0
numpy==1.24.0
```

```bash
# Build & Run
docker build -t ml-api .
docker run -p 5000:5000 ml-api
```

**Why Docker?**
- "Works on my machine" problem SOLVED
- Same environment everywhere
- Easy to scale (Kubernetes)
- Isolated dependencies

## 5️⃣ GIT/GITHUB ESSENTIAL COMMANDS

```bash
# === SETUP ===
git config --global user.name "Your Name"
git config --global user.email "your@email.com"

# === BASIC WORKFLOW ===
git init                    # Initialize new repo
git clone <url>             # Download existing repo
git status                  # Check what changed
git add .                   # Stage ALL changes
git add file.py             # Stage specific file
git commit -m "Add model"   # Save with message
git push origin main        # Upload to GitHub
git pull origin main        # Download latest

# === BRANCHING ===
git branch feature-x        # Create branch
git checkout feature-x      # Switch to branch
git checkout -b feature-y   # Create AND switch
git merge feature-x         # Merge into current
git branch -d feature-x     # Delete branch

# === USEFUL ===
git log --oneline           # View history (compact)
git diff                    # See unstaged changes
git stash                   # Temp save uncommitted work
git stash pop               # Restore stashed work
git reset --hard HEAD~1     # Undo last commit (DANGEROUS!)
```

### .gitignore for ML Projects:
```
__pycache__/
*.pyc
.env
*.pkl
models/
data/
.ipynb_checkpoints/
venv/
```

## 6️⃣ CI/CD (Continuous Integration / Continuous Deployment)

**WHAT:** Automate testing & deployment on every code push.

### GitHub Actions Example:
```yaml
# .github/workflows/ml-pipeline.yml
name: ML Pipeline

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Set up Python
        uses: actions/setup-python@v4
        with:
          python-version: '3.10'
      - name: Install dependencies
        run: pip install -r requirements.txt
      - name: Run tests
        run: pytest tests/
      - name: Check model accuracy
        run: python test_model.py

  deploy:
    needs: test  # Only deploy if tests pass!
    runs-on: ubuntu-latest
    steps:
      - name: Deploy to AWS
        run: echo "Deploying model..."
```

**CI/CD Flow:**
```
Developer pushes code
    ↓
GitHub Actions triggers
    ↓
Run tests (pytest)
    ↓
Check model accuracy > threshold?
    ↓ YES
Build Docker image
    ↓
Deploy to AWS/GCP
    ↓
Monitor performance
```

## 7️⃣ MODEL MONITORING

**WHY:** Models degrade over time! Data changes (data drift).

**What to monitor:**
1. **Data Drift** — Is incoming data different from training data?
2. **Model Drift** — Is accuracy dropping over time?
3. **Prediction Distribution** — Are predictions still reasonable?
4. **Latency** — Is prediction time acceptable?

**Tools:** MLflow, Weights & Biases, Evidently AI, Grafana

**When to retrain:**
- Accuracy drops below threshold
- Data distribution shifts significantly
- New data patterns emerge
- Scheduled (weekly/monthly)

## 8️⃣ DEPLOYMENT INTERVIEW Q&A

**Q: How would you deploy this model?**
→ Save with joblib → Flask/FastAPI API → Docker → AWS/GCP → Monitor

**Q: What is A/B testing?**
→ Deploy 2 models, split traffic 50/50, compare real performance

**Q: What is model drift?**
→ Model performance degrades because real-world data changed from training data

**Q: Batch vs Real-time prediction?**
→ Batch: Process all at once (nightly reports). Real-time: Instant (fraud detection)

**Q: How to handle model versioning?**
→ MLflow, DVC, or simple naming: model_v1.pkl, model_v2.pkl